# 03 — RFM Segmentation + Customer Clustering


This notebook:
1. Loads RFM features and shows descriptive statistics
2. Applies a classic quintile-based RFM baseline segmentation
3. Runs K-Means with Elbow + Silhouette selection
4. Runs DBSCAN as an alternative
5. Selects the champion model via Silhouette and Davies-Bouldin metrics
6. Interprets and names clusters from median R/F/M profiles
7. Persists `data/features/customer_segments.parquet`
8. Registers the champion K-Means model in MLflow Model Registry

**Reference date for Recency**: last transaction date in the dataset (reproducible, never `datetime.today()`).

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# --- path setup so src.* imports resolve from any cwd ---
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent  # coffee_sales/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

from src.features.rfm_features import run as build_rfm, OUTPUT_PATH as RFM_OUTPUT_PATH
from src.models.clustering.rfm import score_rfm
from src.models.clustering.train import train_kmeans, RFM_FEATURE_COLS
from src.models.clustering.evaluate import evaluate_clustering

# --- MLflow tracking URI (relative to project root) ---
MLFLOW_TRACKING_URI = str(PROJECT_ROOT / 'mlruns')
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment('rfm_clustering')

FEATURES_DIR = PROJECT_ROOT / 'data' / 'features'
RFM_PATH     = FEATURES_DIR / 'rfm_features.parquet'
SEGMENTS_PATH = FEATURES_DIR / 'customer_segments.parquet'

print(f'Project root  : {PROJECT_ROOT}')
print(f'MLflow URI    : {MLFLOW_TRACKING_URI}')
print(f'RFM features  : {RFM_PATH}')

---
## Section 1 — Load RFM Features & Descriptive Statistics

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s — %(message)s')

# Build RFM features if not yet generated
if not RFM_PATH.exists():
    print('RFM features not found — running rfm_features.py ...')
    df_rfm = build_rfm(output_path=RFM_PATH)
else:
    df_rfm = pd.read_parquet(RFM_PATH)
    print(f'Loaded RFM features: {df_rfm.shape[0]} customers')

df_rfm.head()

In [ ]:
print('=== Descriptive Statistics ===')
display(df_rfm[['recency_days','frequency','monetary']].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cols = [('recency_days', 'Recency (days)', 'steelblue'),
        ('frequency',    'Frequency (txns)', 'seagreen'),
        ('monetary',     'Monetary (total spend)', 'darkorange')]

for ax, (col, label, color) in zip(axes, cols):
    ax.hist(df_rfm[col], bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(label, fontsize=12)
    ax.set_xlabel(label)
    ax.set_ylabel('Customers')
    ax.axvline(df_rfm[col].median(), color='red', linestyle='--', label=f'Median: {df_rfm[col].median():.1f}')
    ax.legend(fontsize=9)

plt.suptitle('RFM Distributions — Identified Customers', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot of normalised features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pairs = [('recency_norm','frequency_norm'),
         ('recency_norm','monetary_norm'),
         ('frequency_norm','monetary_norm')]

for ax, (x, y) in zip(axes, pairs):
    ax.scatter(df_rfm[x], df_rfm[y], alpha=0.3, s=10, color='steelblue')
    ax.set_xlabel(x); ax.set_ylabel(y)
    ax.set_title(f'{x} vs {y}')

plt.suptitle('RFM Normalised Feature Pairplots', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('\nCorrelation matrix (normalised features):')
display(df_rfm[['recency_norm','frequency_norm','monetary_norm']].corr().round(3))

---
## Section 2 — Baseline: Quintile-Based RFM Segmentation

In [ ]:
df_scored = score_rfm(df_rfm)

segment_counts = df_scored['segment_name'].value_counts()
print('=== Quintile Segments ===')
display(segment_counts.to_frame('count').assign(pct=lambda d: (d['count']/d['count'].sum()*100).round(1)))

fig, ax = plt.subplots(figsize=(8, 4))
colors = {'Champions': '#2ecc71', 'Loyal': '#3498db', 'Potential': '#f39c12',
          'At Risk': '#e67e22', 'Lost': '#e74c3c'}
bars = [segment_counts.get(s, 0) for s in colors]
ax.bar(list(colors.keys()), bars, color=list(colors.values()), edgecolor='white')
ax.set_title('RFM Quintile Segmentation — Customer Distribution', fontsize=13)
ax.set_ylabel('Number of Customers')
for i, v in enumerate(bars):
    ax.text(i, v + 5, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Median R/F/M by quintile segment
profile_q = df_scored.groupby('segment_name')[['recency_days','frequency','monetary']].median().round(1)
print('=== Median RFM Profile by Quintile Segment ===')
display(profile_q)

---
## Section 3 — K-Means: Elbow Curve + Silhouette Score

In [ ]:
X = df_rfm[RFM_FEATURE_COLS].to_numpy()

K_RANGE = range(2, 9)
inertias = []
silhouettes = []
db_scores = []

for k in K_RANGE:
    model_k, labels_k = train_kmeans(df_rfm, k=k)
    metrics_k = evaluate_clustering(X, labels_k)
    inertias.append(model_k.inertia_)
    silhouettes.append(metrics_k['silhouette'])
    db_scores.append(metrics_k['davies_bouldin'])
    print(f'K={k:2d}  inertia={model_k.inertia_:10.2f}  silhouette={metrics_k["silhouette"]:.4f}  davies_bouldin={metrics_k["davies_bouldin"]:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ks = list(K_RANGE)

# Elbow curve
ax1.plot(ks, inertias, 'o-', color='steelblue', linewidth=2, markersize=7)
ax1.set_xlabel('K (number of clusters)')
ax1.set_ylabel('Inertia (within-cluster SSE)')
ax1.set_title('Elbow Curve — K-Means Inertia', fontsize=12)
ax1.set_xticks(ks)
ax1.grid(True, alpha=0.3)

# Silhouette
ax2.plot(ks, silhouettes, 's-', color='seagreen', linewidth=2, markersize=7)
ax2.set_xlabel('K (number of clusters)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score vs K', fontsize=12)
ax2.set_xticks(ks)
ax2.grid(True, alpha=0.3)

# Select K: prefer the K with best silhouette among K>=3 for marketing usability;
# fall back to best Davies-Bouldin if needed.
MIN_MARKETING_K = 3
valid_ks = [(k, sil, db) for k, sil, db in zip(ks, silhouettes, db_scores) if k >= MIN_MARKETING_K]
if valid_ks:
    best_k = max(valid_ks, key=lambda t: t[1])[0]  # best silhouette among K>=3
else:
    best_k_idx = int(np.argmax(silhouettes))
    best_k = ks[best_k_idx]
ax2.axvline(best_k, color='red', linestyle='--', label=f'Best K={best_k}')
ax2.legend()

plt.suptitle('K-Means Model Selection', fontsize=14)
plt.tight_layout()
plt.show()

best_sil = silhouettes[ks.index(best_k)]
print(f'\n→ Best K (silhouette among K≥{MIN_MARKETING_K}): K={best_k} (silhouette={best_sil:.4f})')

In [ ]:
# Train champion K-Means
kmeans_best, labels_kmeans = train_kmeans(df_rfm, k=best_k)
metrics_kmeans = evaluate_clustering(X, labels_kmeans)
print(f'Champion K-Means (K={best_k})')
print(f'  Silhouette     : {metrics_kmeans["silhouette"]:.4f}')
print(f'  Davies-Bouldin : {metrics_kmeans["davies_bouldin"]:.4f}')

---
## Section 4 — DBSCAN Alternative: k-Distance Graph + Evaluation

In [ ]:
# k-distance graph to select eps for DBSCAN
k_nn = 5
nbrs = NearestNeighbors(n_neighbors=k_nn).fit(X)
distances, _ = nbrs.kneighbors(X)
k_distances = np.sort(distances[:, k_nn - 1])[::-1]

plt.figure(figsize=(10, 4))
plt.plot(k_distances, color='steelblue', linewidth=1.5)
plt.xlabel('Points sorted by distance')
plt.ylabel(f'{k_nn}-NN distance')
plt.title(f'k-Distance Graph (k={k_nn}) — DBSCAN eps selection', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Heuristic: pick the knee of the k-distance curve
# Use the 85th percentile of distances as a reasonable eps starting point
eps_candidate = float(np.percentile(k_distances, 15))
print(f'Suggested eps (15th percentile of k-distances): {eps_candidate:.4f}')

In [ ]:
# Try several eps values and pick the one with best Silhouette
eps_values = np.round(np.linspace(eps_candidate * 0.5, eps_candidate * 2.5, 8), 4)
dbscan_results = []

for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5).fit(X)
    labels_db = db.labels_
    n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
    n_noise = (labels_db == -1).sum()
    if n_clusters_db >= 2:
        try:
            m = evaluate_clustering(X, labels_db)
            dbscan_results.append({'eps': eps, 'n_clusters': n_clusters_db,
                                   'n_noise': n_noise, **m})
        except ValueError:
            pass
    else:
        dbscan_results.append({'eps': eps, 'n_clusters': n_clusters_db,
                               'n_noise': n_noise, 'silhouette': np.nan, 'davies_bouldin': np.nan})

df_dbscan_results = pd.DataFrame(dbscan_results)
print('=== DBSCAN Parameter Search ===')
display(df_dbscan_results.round(4))

# Select best DBSCAN config by silhouette
valid_db = df_dbscan_results.dropna(subset=['silhouette'])
if len(valid_db) > 0:
    best_db_row = valid_db.loc[valid_db['silhouette'].idxmax()]
    best_eps = best_db_row['eps']
    print(f'\nBest DBSCAN eps={best_eps:.4f}  clusters={int(best_db_row["n_clusters"])}  silhouette={best_db_row["silhouette"]:.4f}')
else:
    best_eps = eps_candidate
    print('No valid DBSCAN config found — using candidate eps')

In [ ]:
# Fit best DBSCAN
dbscan_best = DBSCAN(eps=best_eps, min_samples=5).fit(X)
labels_dbscan = dbscan_best.labels_
n_clusters_dbscan = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
n_noise_dbscan = (labels_dbscan == -1).sum()

print(f'DBSCAN (eps={best_eps:.4f}, min_samples=5)')
print(f'  Clusters found : {n_clusters_dbscan}')
print(f'  Noise points   : {n_noise_dbscan} ({n_noise_dbscan/len(labels_dbscan)*100:.1f}%)')

if n_clusters_dbscan >= 2:
    metrics_dbscan = evaluate_clustering(X, labels_dbscan)
    print(f'  Silhouette     : {metrics_dbscan["silhouette"]:.4f}')
    print(f'  Davies-Bouldin : {metrics_dbscan["davies_bouldin"]:.4f}')
else:
    metrics_dbscan = {'silhouette': np.nan, 'davies_bouldin': np.nan}
    print('  Not enough clusters for metric evaluation')

---
## Section 5 — Champion Model Selection + MLflow Logging

In [ ]:
# Compare K-Means vs DBSCAN
comparison = pd.DataFrame([
    {'model': f'K-Means (K={best_k})', 'silhouette': metrics_kmeans['silhouette'],
     'davies_bouldin': metrics_kmeans['davies_bouldin'],
     'n_clusters': best_k, 'n_noise': 0},
    {'model': f'DBSCAN (eps={best_eps:.4f})', 'silhouette': metrics_dbscan['silhouette'],
     'davies_bouldin': metrics_dbscan['davies_bouldin'],
     'n_clusters': n_clusters_dbscan, 'n_noise': n_noise_dbscan},
])
print('=== Model Comparison ===')
display(comparison.round(4))

# Champion selection:
# - Disqualify DBSCAN if noise ratio > 50% (not practically useful for segmentation)
# - Higher silhouette wins among qualified models
kmeans_sil = metrics_kmeans['silhouette']
noise_ratio = n_noise_dbscan / len(df_rfm) if len(df_rfm) > 0 else 1.0
dbscan_qualified = (not np.isnan(metrics_dbscan['silhouette'])) and (noise_ratio <= 0.5)
dbscan_sil = metrics_dbscan['silhouette'] if dbscan_qualified else -999
champion = 'kmeans' if kmeans_sil >= dbscan_sil else 'dbscan'
print(f'  DBSCAN noise ratio: {noise_ratio:.1%} — qualified: {dbscan_qualified}')
print(f'\n→ Champion model: {champion.upper()}')

In [ ]:
# Log K-Means run
with mlflow.start_run(run_name=f'kmeans_k{best_k}') as run_km:
    mlflow.log_param('model_type', 'kmeans')
    mlflow.log_param('k', best_k)
    mlflow.log_param('n_customers', len(df_rfm))
    mlflow.log_param('features', ','.join(RFM_FEATURE_COLS))
    mlflow.log_metric('silhouette', metrics_kmeans['silhouette'])
    mlflow.log_metric('davies_bouldin', metrics_kmeans['davies_bouldin'])
    mlflow.log_metric('inertia', float(kmeans_best.inertia_))
    mlflow.set_tag('champion', str(champion == 'kmeans'))
    mlflow.sklearn.log_model(kmeans_best, artifact_path='model')
    km_run_id = run_km.info.run_id

print(f'K-Means run logged: {km_run_id}')

# Log DBSCAN run (metrics only — DBSCAN model not logged to registry)
with mlflow.start_run(run_name=f'dbscan_eps{best_eps:.4f}') as run_db:
    mlflow.log_param('model_type', 'dbscan')
    mlflow.log_param('eps', best_eps)
    mlflow.log_param('min_samples', 5)
    mlflow.log_param('n_customers', len(df_rfm))
    mlflow.log_metric('n_clusters', n_clusters_dbscan)
    mlflow.log_metric('n_noise', int(n_noise_dbscan))
    if not np.isnan(metrics_dbscan['silhouette']):
        mlflow.log_metric('silhouette', metrics_dbscan['silhouette'])
        mlflow.log_metric('davies_bouldin', metrics_dbscan['davies_bouldin'])
    mlflow.set_tag('champion', str(champion == 'dbscan'))
    db_run_id = run_db.info.run_id

print(f'DBSCAN  run logged: {db_run_id}')

---
## Section 6 — Cluster Interpretation & Naming

In [ ]:
# Attach K-Means labels to RFM frame for analysis
df_clustered = df_rfm.copy()
df_clustered['cluster_id'] = labels_kmeans

# Median R/F/M by cluster
cluster_profiles = df_clustered.groupby('cluster_id')[['recency_days','frequency','monetary']].median().round(1)
cluster_sizes    = df_clustered.groupby('cluster_id').size().rename('size')
cluster_summary  = cluster_profiles.join(cluster_sizes)
print('=== Cluster Median Profiles ===')
display(cluster_summary)

In [ ]:
# --- Cluster naming: rank clusters by median recency_days, assign names from ladder ---
# Sorted from most-recent (best) to least-recent (worst).
# The most-recent cluster with the highest frequency gets 'Champions'; others follow the ladder.
# This always produces K unique names regardless of cluster profiles.

_NAME_LADDER = ['Champions', 'Loyal', 'Potential', 'At Risk', 'Lost', 'Inactive']

# Sort cluster_ids by recency_days ascending (most recent first)
sorted_cids = cluster_summary['recency_days'].sort_values(ascending=True).index.tolist()

# If the most-recent cluster also has the highest frequency, call it Champions;
# otherwise call it Loyal (still active but not the highest engagement)
most_recent_cid = sorted_cids[0]
max_freq_cid    = cluster_summary['frequency'].idxmax()
if most_recent_cid != max_freq_cid:
    # Swap so Champions goes to highest-freq cluster if it's among the top-2 recent
    top2 = sorted_cids[:2]
    if max_freq_cid in top2:
        sorted_cids[top2.index(most_recent_cid)], sorted_cids[top2.index(max_freq_cid)] = (
            sorted_cids[top2.index(max_freq_cid)], sorted_cids[top2.index(most_recent_cid)]
        )

segment_map = {cid: _NAME_LADDER[min(i, len(_NAME_LADDER)-1)] for i, cid in enumerate(sorted_cids)}
cluster_summary['segment_name'] = cluster_summary.index.map(segment_map)

print('=== Cluster Names ===')
display(cluster_summary[['recency_days','frequency','monetary','size','segment_name']])

# Map cluster_id → segment_name
CLUSTER_NAMES = cluster_summary['segment_name'].to_dict()
df_clustered['segment_name'] = df_clustered['cluster_id'].map(CLUSTER_NAMES)
print('\nSegment distribution:')
display(df_clustered['segment_name'].value_counts())

In [ ]:
# Interactive scatter: Recency vs Frequency, coloured by Monetary
fig = px.scatter(
    df_clustered,
    x='recency_days',
    y='frequency',
    color='segment_name',
    size='monetary',
    size_max=20,
    hover_data=['card', 'monetary'],
    opacity=0.7,
    title='Customer Segments — Recency vs Frequency (size = Monetary)',
    labels={'recency_days': 'Recency (days)', 'frequency': 'Frequency (transactions)', 'segment_name': 'Segment'},
    template='plotly_white',
    width=900, height=550,
)

# Add centroids
centroids = kmeans_best.cluster_centers_  # shape (k, 3): recency_norm, freq_norm, mon_norm
# Back-transform centroids to original scale for display
r_min, r_max = df_rfm['recency_days'].min(), df_rfm['recency_days'].max()
f_min, f_max = df_rfm['frequency'].min(), df_rfm['frequency'].max()

for i, (cid, row) in enumerate(cluster_summary.iterrows()):
    # centroid recency_norm is inverted: recency_days = r_max - recency_norm*(r_max-r_min)
    c_recency = r_max - centroids[cid, 0] * (r_max - r_min)
    c_freq    = f_min + centroids[cid, 1] * (f_max - f_min)
    fig.add_trace(go.Scatter(
        x=[c_recency], y=[c_freq],
        mode='markers+text',
        marker=dict(symbol='x', size=14, color='black', line=dict(width=2)),
        text=[f"★ {row['segment_name']}"], textposition='top center',
        showlegend=False,
    ))

fig.show()

---
## Section 7 — Save `customer_segments.parquet`

In [ ]:
df_segments = df_clustered[['card','recency_days','frequency','monetary','cluster_id','segment_name']].copy()

SEGMENTS_PATH.parent.mkdir(parents=True, exist_ok=True)
df_segments.to_parquet(SEGMENTS_PATH, index=False, engine='pyarrow')
print(f'Saved customer_segments.parquet: {SEGMENTS_PATH}')
print(f'Shape: {df_segments.shape}')
display(df_segments.head())
display(df_segments['segment_name'].value_counts())

---
## Section 8 — Register Champion K-Means in MLflow Model Registry

In [ ]:
MODEL_NAME = 'rfm_clustering_kmeans'
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

# Register model from the K-Means run
model_uri = f'runs:/{km_run_id}/model'
mv = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)
print(f'Registered model: {MODEL_NAME}  version={mv.version}')

# Transition to Production
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=mv.version,
    stage='Production',
    archive_existing_versions=True,
)
print(f'Model {MODEL_NAME} v{mv.version} → Production')

# Tag the champion run
client.set_tag(km_run_id, 'registered_model', MODEL_NAME)
client.set_tag(km_run_id, 'stage', 'Production')

# Verify
prod_versions = client.get_latest_versions(MODEL_NAME, stages=['Production'])
for v in prod_versions:
    print(f'  ✓ {v.name}  version={v.version}  stage={v.current_stage}  run_id={v.run_id}')

In [ ]:
print(f'  RFM features      → {RFM_PATH}')
print(f'  Customer segments → {SEGMENTS_PATH}')
print(f'  MLflow experiment : rfm_clustering')
print(f'  Model Registry    : {MODEL_NAME} @ Production')
print(f'  K-Means run id    : {km_run_id}')
print(f'  DBSCAN  run id    : {db_run_id}')
print(f'  Best K            : {best_k}')
print(f'  Silhouette Score  : {metrics_kmeans["silhouette"]:.4f}')
print(f'  Davies-Bouldin    : {metrics_kmeans["davies_bouldin"]:.4f}')
print(f'  Customers segmented: {len(df_segments)}')